In [1]:
import pandas as pd
import math
import numpy as np
from typing import Optional

In [2]:
df = pd.read_csv("water_leak_detection_1000_rows 3.csv")

df.head()

,timestamp,sensor_id,pressure_bar,flow_rate_L_s,temperature_c,Leak Status,Burst Status,leak
0,2024-01-01 0:00,S007,3.694814,77.515218,21.695365,0,0,0
1,2024-01-01 0:05,S007,2.587125,179.926422,19.016725,0,0,0
2,2024-01-01 0:10,S002,2.448965,210.130823,10.011681,1,0,1
3,2024-01-01 0:15,S009,2.936844,141.777934,12.092408,0,0,0
4,2024-01-01 0:20,S003,3.073693,197.484633,17.000000,0,0,0


In [3]:
df.drop(columns=['Leak Status', 'Burst Status'], inplace=True)

In [4]:
df.head()

,timestamp,sensor_id,pressure_bar,flow_rate_L_s,temperature_c,leak
0,2024-01-01 0:00,S007,3.694814,77.515218,21.695365,0
1,2024-01-01 0:05,S007,2.587125,179.926422,19.016725,0
2,2024-01-01 0:10,S002,2.448965,210.130823,10.011681,1
3,2024-01-01 0:15,S009,2.936844,141.777934,12.092408,0
4,2024-01-01 0:20,S003,3.073693,197.484633,17.000000,0


In [5]:
df['pipe_diameter_m'] = 0.1524
df['velocity'] = (df['flow_rate_L_s'] / 1000) / (df['pipe_diameter_m']**2 * np.pi / 4)

In [6]:
df['speed_of_sound'] = (
    1402.388 + 5.03830*df['temperature_c'] - 5.81090e-2*(df['temperature_c']**2) + 
    3.3432e-4*(df['temperature_c']**3) - 1.47797e-6*(df['temperature_c']**4) + 3.1419e-9*(df['temperature_c']**5)
)

In [7]:
df['angle_rad'] = np.deg2rad(45)
df['traverses'] = 2

In [8]:
df['l_path'] = df['traverses'] *  df['pipe_diameter_m']/ np.sin(df['angle_rad'])

In [9]:
df['t_down'] = df['l_path'] / (df['speed_of_sound'] + df['velocity'] * np.cos(df['angle_rad']))
df['t_up'] = df['l_path'] / (df['speed_of_sound'] - df['velocity'] * np.cos(df['angle_rad']))

df['delta_t'] = df['t_down'] - df['t_up']

In [10]:
df.head()

,timestamp,sensor_id,pressure_bar,flow_rate_L_s,temperature_c,leak,pipe_diameter_m,velocity,speed_of_sound,angle_rad,traverses,l_path,t_down,t_up,delta_t
0,2024-01-01 0:00,S007,3.694814,77.515218,21.695365,0,0.1524,4.249396,1487.446157,0.785398,2,0.431052,0.000289,0.000290,-0.000001
1,2024-01-01 0:05,S007,2.587125,179.926422,19.016725,0,0.1524,9.863593,1479.299356,0.785398,2,0.431052,0.000290,0.000293,-0.000003
2,2024-01-01 0:10,S002,2.448965,210.130823,10.011681,1,0.1524,11.519402,1447.326331,0.785398,2,0.431052,0.000296,0.000300,-0.000003
3,2024-01-01 0:15,S009,2.936844,141.777934,12.092408,0,0.1524,7.772287,1455.376477,0.785398,2,0.431052,0.000295,0.000297,-0.000002
4,2024-01-01 0:20,S003,3.073693,197.484633,17.000000,0,0.1524,10.826136,1472.769133,0.785398,2,0.431052,0.000291,0.000294,-0.000003


## Scalling

In [11]:
df['pipe_diameter_m_scaled'] = df['pipe_diameter_m'] / 6

In [12]:
# Temperature
df['temp_c_scaled'] = (df['temperature_c'] -  df['temperature_c'].mean()) * (1.0 / df['temperature_c'].std()) + 15

In [13]:
target_p95_v = 1.3 # m/s (choose 0.8–1.5; 2.0 is a “very high” residential p95)

beta = target_p95_v / df['velocity'].quantile(0.95)  # dimensionless

# 1) Scale velocity distribution
df['velocity_scaled'] = beta * df['velocity']

In [14]:
# 2) Compute flow from velocity and target diameter
A1 = (np.pi/4.0) * (df['pipe_diameter_m_scaled']**2)      # m²
df['flow_L_s_scaled'] = df['velocity_scaled'] * A1 * 1000 # L/s

In [15]:
df['speed_of_sound_scaled'] = (
    1402.388 + 5.03830*df['temp_c_scaled'] - 5.81090e-2*(df['temp_c_scaled']**2) + 
    3.3432e-4*(df['temp_c_scaled']**3) - 1.47797e-6*(df['temp_c_scaled']**4) + 3.1419e-9*(df['temp_c_scaled']**5)
)

In [16]:
df['l_path_scaled'] = df['traverses'] *  df['pipe_diameter_m_scaled']/ np.sin(df['angle_rad'])

In [17]:
df.head()

,timestamp,sensor_id,pressure_bar,flow_rate_L_s,temperature_c,leak,pipe_diameter_m,velocity,speed_of_sound,angle_rad,...,l_path,t_down,t_up,delta_t,pipe_diameter_m_scaled,temp_c_scaled,velocity_scaled,flow_L_s_scaled,speed_of_sound_scaled,l_path_scaled
0,2024-01-01 0:00,S007,3.694814,77.515218,21.695365,0,0.1524,4.249396,1487.446157,0.785398,...,0.431052,0.000289,0.000290,-0.000001,0.0254,15.993382,0.520785,0.263885,1469.378119,0.071842
1,2024-01-01 0:05,S007,2.587125,179.926422,19.016725,0,0.1524,9.863593,1479.299356,0.785398,...,0.431052,0.000290,0.000293,-0.000003,0.0254,15.368837,1.208832,0.612524,1467.229261,0.071842
2,2024-01-01 0:10,S002,2.448965,210.130823,10.011681,1,0.1524,11.519402,1447.326331,0.785398,...,0.431052,0.000296,0.000300,-0.000003,0.0254,13.269243,1.411760,0.715349,1459.747573,0.071842
3,2024-01-01 0:15,S009,2.936844,141.777934,12.092408,0,0.1524,7.772287,1455.376477,0.785398,...,0.431052,0.000295,0.000297,-0.000002,0.0254,13.754380,0.952532,0.482655,1461.512041,0.071842
4,2024-01-01 0:20,S003,3.073693,197.484633,17.000000,0,0.1524,10.826136,1472.769133,0.785398,...,0.431052,0.000291,0.000294,-0.000003,0.0254,14.898622,1.326797,0.672298,1465.588427,0.071842


In [18]:
df['t_down_scaled'] = df['l_path_scaled'] / (df['speed_of_sound_scaled'] + df['velocity_scaled'] * np.cos(df['angle_rad']))
df['t_up_scaled'] = df['l_path_scaled'] / (df['speed_of_sound_scaled'] - df['velocity_scaled'] * np.cos(df['angle_rad']))

df['delta_t_scaled'] = df['t_down_scaled'] - df['t_up_scaled']

In [19]:
def estimate_static_pressure_bar(df, flow_col='flow_rate_L_s', pres_col='pressure_bar',
                                 low_flow_thresh=0.05, percentile=5):
    """
    Estimate static (supply) pressure as a low-percentile of pressure
    during near-zero flow. Falls back to the same percentile over all data.
    """
    low_flow = df[flow_col].abs() <= low_flow_thresh
    if low_flow.any():
        return np.percentile(df.loc[low_flow, pres_col], percentile)
    return np.percentile(df[pres_col], percentile)

In [20]:
def scale_pressure_to_residential(
    df: pd.DataFrame,
    *,
    # Target static pressure (e.g., 60 psi ≈ 4.137 bar)
    p_set_bar: float = 3.45,
    # Geometry/loss parameters (set what you know; others can stay None/0)
    # Original setup:
    f0: Optional[float] = 0.022,     # Darcy friction factor (original)
    L0_m: Optional[float] = 1000,   # Length of influential run (original)
    sumK0: float = 0.0,          # Sum of minor-loss Ks near the sensor (original)
    # Target residential setup:
    f1: Optional[float] = 0.027,     # Darcy friction factor (residential)
    L1_m: Optional[float] = 8,   # Length of influential run (residential)
    sumK1: float = 4.0,          # Typical main valve + meter + PRV + a couple of elbows
    # PRV droop (optional): bar per (L/s)
    prv_droop_bar_per_Lps: float = 0.0,
    # Static estimation parameters
    low_flow_thresh: float = 0.05,
    static_percentile: int = 5,
    # Debug columns
    write_debug_cols: bool = True
) -> pd.DataFrame:
    """
    Scales df['pressure_bar'] -> df['pressure_bar_scaled'] using:
      p1(t) = p_set_bar + [p0(t) - p_static0_bar] * (C1/C0) * (v1/v0)^2 - S_d * Q1(t)
    where C = f*L/D + sumK, v from Q and D.
    Uses these columns (already in your frame):
      - flow_rate_L_s, pipe_diameter_m, pressure_bar  (original)
      - flow_L_s_scaled, pipe_diameter_m_scaled       (target)
    """
    df = df.copy()

    # --- 1) Velocities (use your columns) ---
    v0 = df['velocity']
    v1 = df['velocity_scaled']

    # --- 2) Static/dynamic split for original pressure ---
    p_static0_bar = estimate_static_pressure_bar(
        df, flow_col='flow_rate_L_s', pres_col='pressure_bar',
        low_flow_thresh=low_flow_thresh, percentile=static_percentile
    )
    dp0_bar = df['pressure_bar'].to_numpy() - p_static0_bar  # dynamic component (can be small/negative)

    # --- 3) Build C0 and C1 = f*L/D + sumK ---
    D0 = df['pipe_diameter_m'].to_numpy()
    D1 = df['pipe_diameter_m_scaled'].to_numpy()

    # start with sumK terms
    C0 = np.full(len(df), float(sumK0))
    C1 = np.full(len(df), float(sumK1))

    # add friction terms if given
    if f0 is not None and L0_m is not None:
        with np.errstate(divide='ignore', invalid='ignore'):
            C0 = C0 + f0 * (L0_m / np.where(D0 > 0, D0, np.nan))
            C0 = np.nan_to_num(C0, nan=float(sumK0))
    if f1 is not None and L1_m is not None:
        with np.errstate(divide='ignore', invalid='ignore'):
            C1 = C1 + f1 * (L1_m / np.where(D1 > 0, D1, np.nan))
            C1 = np.nan_to_num(C1, nan=float(sumK1))

    # guard against zero/negative C0
    eps = 1e-12
    C0_safe = np.where(C0 > eps, C0, eps)

    # --- 4) Scale dynamic pressure ---
    vel_ratio2 = np.where(np.abs(v0) > eps, (v1 / v0)**2, 0.0)
    geom_ratio = C1 / C0_safe

    dp1_bar = dp0_bar * geom_ratio * vel_ratio2

    # --- 5) Add target static and PRV droop ---
    p1_bar = p_set_bar + dp1_bar - prv_droop_bar_per_Lps * df['flow_L_s_scaled'].to_numpy()

    # --- 6) Write results ---
    df['pressure_bar_scaled'] = p1_bar

    if write_debug_cols:
        df['pressure_bar_static_est']      = p_static0_bar
        df['pressure_bar_dynamic']         = dp0_bar
        df['pressure_bar_dynamic_scaled']  = dp1_bar
        df['velocity']                     = v0  # overwrite/refresh from Q,D (optional)
        df['velocity_scaled']              = v1
        df['C0_effective']                 = C0
        df['C1_effective']                 = C1
        df['geom_ratio_C1_over_C0']        = geom_ratio
        df['vel_ratio_sq']                 = vel_ratio2

    return df

In [21]:
df_scaled = scale_pressure_to_residential(
    df,
    p_set_bar=3.45,     # ~60 psi
    sumK0=1.0,           # guess for original near-sensor minor losses (adjust if you know it)
    sumK1=4.0,           # main valve + meter + PRV + a couple of elbows (tune to taste)
    prv_droop_bar_per_Lps=0.0  # set e.g. 0.02–0.08 if you want visible PRV droop
)


In [22]:
df_scaled.head()

,timestamp,sensor_id,pressure_bar,flow_rate_L_s,temperature_c,leak,pipe_diameter_m,velocity,speed_of_sound,angle_rad,...,t_up_scaled,delta_t_scaled,pressure_bar_scaled,pressure_bar_static_est,pressure_bar_dynamic,pressure_bar_dynamic_scaled,C0_effective,C1_effective,geom_ratio_C1_over_C0,vel_ratio_sq
0,2024-01-01 0:00,S007,3.694814,77.515218,21.695365,0,0.1524,4.249396,1487.446157,0.785398,...,0.000049,-2.450669e-08,3.451500,2.533621,1.161194,0.001500,145.356955,12.503937,0.086022,0.01502
1,2024-01-01 0:05,S007,2.587125,179.926422,19.016725,0,0.1524,9.863593,1479.299356,0.785398,...,0.000049,-5.705110e-08,3.450069,2.533621,0.053505,0.000069,145.356955,12.503937,0.086022,0.01502
2,2024-01-01 0:10,S002,2.448965,210.130823,10.011681,1,0.1524,11.519402,1447.326331,0.785398,...,0.000049,-6.731305e-08,3.449891,2.533621,-0.084656,-0.000109,145.356955,12.503937,0.086022,0.01502
3,2024-01-01 0:15,S009,2.936844,141.777934,12.092408,0,0.1524,7.772287,1455.376477,0.785398,...,0.000049,-4.530736e-08,3.450521,2.533621,0.403223,0.000521,145.356955,12.503937,0.086022,0.01502
4,2024-01-01 0:20,S003,3.073693,197.484633,17.000000,0,0.1524,10.826136,1472.769133,0.785398,...,0.000049,-6.275875e-08,3.450698,2.533621,0.540072,0.000698,145.356955,12.503937,0.086022,0.01502


In [23]:
df_scaled['pressure_bar_scaled'].describe()

count    1000.000000
mean        3.450888
std         0.000632
min         3.447903
25%         3.450421
50%         3.450946
75%         3.451387
max         3.451889
Name: pressure_bar_scaled, dtype: float64

In [24]:
df_scaled['pressure_bar'].describe()

count    1000.000000
mean        3.220690
std         0.489030
min         0.910977
25%         2.859332
50%         3.265711
75%         3.607196
max         3.995364
Name: pressure_bar, dtype: float64

In [25]:
df_scaled['velocity_scaled'].describe()

count    1000.000000
mean        0.840066
std         0.296429
min         0.340321
25%         0.590869
50%         0.833810
75%         1.088977
max         2.228884
Name: velocity_scaled, dtype: float64

In [26]:
df_scaled['flow_L_s_scaled'].describe()

count    1000.000000
mean        0.425668
std         0.150203
min         0.172443
25%         0.299398
50%         0.422498
75%         0.551793
max         1.129392
Name: flow_L_s_scaled, dtype: float64

In [27]:
df_scaled.columns

Index(['timestamp', 'sensor_id', 'pressure_bar', 'flow_rate_L_s',
       'temperature_c', 'leak', 'pipe_diameter_m', 'velocity',
       'speed_of_sound', 'angle_rad', 'traverses', 'l_path', 't_down', 't_up',
       'delta_t', 'pipe_diameter_m_scaled', 'temp_c_scaled', 'velocity_scaled',
       'flow_L_s_scaled', 'speed_of_sound_scaled', 'l_path_scaled',
       't_down_scaled', 't_up_scaled', 'delta_t_scaled', 'pressure_bar_scaled',
       'pressure_bar_static_est', 'pressure_bar_dynamic',
       'pressure_bar_dynamic_scaled', 'C0_effective', 'C1_effective',
       'geom_ratio_C1_over_C0', 'vel_ratio_sq'],
      dtype='object')

In [28]:
df_scaled.head()

,timestamp,sensor_id,pressure_bar,flow_rate_L_s,temperature_c,leak,pipe_diameter_m,velocity,speed_of_sound,angle_rad,...,t_up_scaled,delta_t_scaled,pressure_bar_scaled,pressure_bar_static_est,pressure_bar_dynamic,pressure_bar_dynamic_scaled,C0_effective,C1_effective,geom_ratio_C1_over_C0,vel_ratio_sq
0,2024-01-01 0:00,S007,3.694814,77.515218,21.695365,0,0.1524,4.249396,1487.446157,0.785398,...,0.000049,-2.450669e-08,3.451500,2.533621,1.161194,0.001500,145.356955,12.503937,0.086022,0.01502
1,2024-01-01 0:05,S007,2.587125,179.926422,19.016725,0,0.1524,9.863593,1479.299356,0.785398,...,0.000049,-5.705110e-08,3.450069,2.533621,0.053505,0.000069,145.356955,12.503937,0.086022,0.01502
2,2024-01-01 0:10,S002,2.448965,210.130823,10.011681,1,0.1524,11.519402,1447.326331,0.785398,...,0.000049,-6.731305e-08,3.449891,2.533621,-0.084656,-0.000109,145.356955,12.503937,0.086022,0.01502
3,2024-01-01 0:15,S009,2.936844,141.777934,12.092408,0,0.1524,7.772287,1455.376477,0.785398,...,0.000049,-4.530736e-08,3.450521,2.533621,0.403223,0.000521,145.356955,12.503937,0.086022,0.01502
4,2024-01-01 0:20,S003,3.073693,197.484633,17.000000,0,0.1524,10.826136,1472.769133,0.785398,...,0.000049,-6.275875e-08,3.450698,2.533621,0.540072,0.000698,145.356955,12.503937,0.086022,0.01502


## Residential data

In [29]:
residential_df = df_scaled.drop(columns=['pressure_bar', 'flow_rate_L_s', 'temperature_c', 
       'pipe_diameter_m', 'velocity', 'speed_of_sound', 'l_path','t_down', 't_up', 
       'delta_t', 'pressure_bar_static_est', 'pressure_bar_dynamic',
       'C0_effective', 'C1_effective', 'geom_ratio_C1_over_C0', 'vel_ratio_sq'])

residential_df.head()

,timestamp,sensor_id,leak,angle_rad,traverses,pipe_diameter_m_scaled,temp_c_scaled,velocity_scaled,flow_L_s_scaled,speed_of_sound_scaled,l_path_scaled,t_down_scaled,t_up_scaled,delta_t_scaled,pressure_bar_scaled,pressure_bar_dynamic_scaled
0,2024-01-01 0:00,S007,0,0.785398,2,0.0254,15.993382,0.520785,0.263885,1469.378119,0.071842,0.000049,0.000049,-2.450669e-08,3.451500,0.001500
1,2024-01-01 0:05,S007,0,0.785398,2,0.0254,15.368837,1.208832,0.612524,1467.229261,0.071842,0.000049,0.000049,-5.705110e-08,3.450069,0.000069
2,2024-01-01 0:10,S002,1,0.785398,2,0.0254,13.269243,1.411760,0.715349,1459.747573,0.071842,0.000049,0.000049,-6.731305e-08,3.449891,-0.000109
3,2024-01-01 0:15,S009,0,0.785398,2,0.0254,13.754380,0.952532,0.482655,1461.512041,0.071842,0.000049,0.000049,-4.530736e-08,3.450521,0.000521
4,2024-01-01 0:20,S003,0,0.785398,2,0.0254,14.898622,1.326797,0.672298,1465.588427,0.071842,0.000049,0.000049,-6.275875e-08,3.450698,0.000698


In [30]:
residential_df['timestamp'] = pd.to_datetime(residential_df['timestamp'], utc=False)

leak_times = residential_df.loc[residential_df['leak'].eq(1), ['timestamp']].rename(columns={'timestamp':'leak_time'})
next_leak = pd.merge_asof(
    residential_df[['timestamp']], leak_times, left_on='timestamp', right_on='leak_time',
    direction='forward'
)
delta = next_leak['leak_time'] - residential_df['timestamp']
residential_df['leak_next_hour'] = ((delta > pd.Timedelta(0)) & (delta <= pd.Timedelta('60min'))).fillna(False).astype(int)

residential_df.head()

,timestamp,sensor_id,leak,angle_rad,traverses,pipe_diameter_m_scaled,temp_c_scaled,velocity_scaled,flow_L_s_scaled,speed_of_sound_scaled,l_path_scaled,t_down_scaled,t_up_scaled,delta_t_scaled,pressure_bar_scaled,pressure_bar_dynamic_scaled,leak_next_hour
0,2024-01-01 00:00:00,S007,0,0.785398,2,0.0254,15.993382,0.520785,0.263885,1469.378119,0.071842,0.000049,0.000049,-2.450669e-08,3.451500,0.001500,1
1,2024-01-01 00:05:00,S007,0,0.785398,2,0.0254,15.368837,1.208832,0.612524,1467.229261,0.071842,0.000049,0.000049,-5.705110e-08,3.450069,0.000069,1
2,2024-01-01 00:10:00,S002,1,0.785398,2,0.0254,13.269243,1.411760,0.715349,1459.747573,0.071842,0.000049,0.000049,-6.731305e-08,3.449891,-0.000109,0
3,2024-01-01 00:15:00,S009,0,0.785398,2,0.0254,13.754380,0.952532,0.482655,1461.512041,0.071842,0.000049,0.000049,-4.530736e-08,3.450521,0.000521,0
4,2024-01-01 00:20:00,S003,0,0.785398,2,0.0254,14.898622,1.326797,0.672298,1465.588427,0.071842,0.000049,0.000049,-6.275875e-08,3.450698,0.000698,0


In [31]:
residential_df.columns

Index(['timestamp', 'sensor_id', 'leak', 'angle_rad', 'traverses',
       'pipe_diameter_m_scaled', 'temp_c_scaled', 'velocity_scaled',
       'flow_L_s_scaled', 'speed_of_sound_scaled', 'l_path_scaled',
       't_down_scaled', 't_up_scaled', 'delta_t_scaled', 'pressure_bar_scaled',
       'pressure_bar_dynamic_scaled', 'leak_next_hour'],
      dtype='object')

## Model

In [32]:
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import TCNModel
from darts.utils.likelihood_models.torch import BernoulliLikelihood

from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score

import torch

# Disable MPS to force CPU usage
torch.backends.mps.is_available = lambda: False

/opt/anaconda3/envs/WDN_sim/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
FEATURE_COLS = [
    "pipe_diameter_m_scaled", "temp_c_scaled", "velocity_scaled",
    "flow_L_s_scaled", "speed_of_sound_scaled", "l_path_scaled", "t_down_scaled", 
    "t_up_scaled", "delta_t_scaled", "pressure_bar_scaled", "pressure_bar_dynamic_scaled"
]

In [34]:
residential_df = residential_df.sort_values("timestamp").copy()

residential_df[["leak_next_hour"] + FEATURE_COLS] = (
    residential_df[["leak_next_hour"] + FEATURE_COLS].astype("float32")
)

residential_df["leak_next_hour"] = residential_df["leak_next_hour"].astype(int)

In [35]:
series = TimeSeries.from_dataframe(residential_df, time_col="timestamp", value_cols="leak_next_hour")
covs   = TimeSeries.from_dataframe(residential_df, time_col="timestamp", value_cols=FEATURE_COLS)

In [36]:
split_fraction = 0.8
split_index = int(len(series) * split_fraction)

train_target, val_target = series[:split_index], series[split_index:]
train_covs, val_covs = covs[:split_index], covs[split_index:]

In [37]:
input_chunk_length = 48  # adjust for your data freq
output_chunk_length = 1  # next hour
assert len(train_target) > input_chunk_length, "Train set too short for chosen input_chunk_length."

In [38]:
cov_scaler = Scaler()
covs_sc = cov_scaler.fit_transform(covs)

# targets
train_target = series[:split_index]
val_target   = series[split_index:]

# covariates for training
train_covs_sc = covs_sc[:split_index]

# covariates for validation: extend back to provide context for input_chunk_length
val_start_extended = max(0, split_index - input_chunk_length + 1)
val_covs_sc = covs_sc[val_start_extended:]

In [39]:
model = TCNModel(
    input_chunk_length=input_chunk_length,
    output_chunk_length=output_chunk_length,
    kernel_size=3,
    num_filters=8,
    dropout=0.2,
    dilation_base=2,
    weight_norm=True,
    n_epochs=50,
    batch_size=32,
    likelihood=BernoulliLikelihood(),
    random_state=0,
    force_reset=True,
    pl_trainer_kwargs={"accelerator": "cpu"}  # Explicitly use CPU
)

# Verify the likelihood is set correctly
print("Model likelihood:", model.likelihood)
print("Model likelihood type:", type(model.likelihood))

Model likelihood: BernoulliLikelihood(prior_p=None, prior_strength=1.0)
Model likelihood type: <class 'darts.utils.likelihood_models.torch.BernoulliLikelihood'>


In [40]:
# Build oversampled training segments around positives
h = input_chunk_length
y_df = residential_df.set_index('timestamp')
y_series = series
cov_series = train_covs_sc

# indices for the training split
split_index = int(len(y_series) * 0.8)
train_times = y_series.time_index[:split_index]
y_train = y_series[:split_index]
cov_train = cov_series[:split_index]

# positive times
pos_times = train_times[y_train.values().reshape(-1) == 1]
# optional label dilation: include a few steps before positives as positives
dilate_steps = 2  # 10 minutes at 5-min resolution
dilated = set()
for t in pos_times:
    idx = train_times.get_loc(t)
    for k in range(max(0, idx - dilate_steps), idx + 1):
        dilated.add(train_times[k])
pos_dilated = sorted(dilated)

# sample negatives far from any positive
pos_set = set(pos_dilated)
neg_times_all = [t for t in train_times if t not in pos_set]
# subsample negatives to reach ~30% positives
import random
random.seed(0)
neg_keep = random.sample(neg_times_all, k=min(len(neg_times_all), int(len(pos_dilated) * 2)))

def cut(ts, center_time, h):
    end = center_time
    start_idx = ts.time_index.get_loc(end) - h  # Remove the +1
    if start_idx < 0: 
        return None
    start = ts.time_index[start_idx]
    return ts[start:end]

train_y_segments = []
train_cov_segments = []

for t in pos_dilated + neg_keep:
    y_seg = cut(y_train, t, h)
    c_seg = cut(cov_train, t, h)
    if y_seg is not None and c_seg is not None:
        train_y_segments.append(y_seg)
        train_cov_segments.append(c_seg)

# validation as a single contiguous sequence is fine
val_target = series[split_index:]

In [41]:
model.fit(
    series=train_y_segments,
    past_covariates=train_cov_segments,
    val_series=val_target,
    val_past_covariates=val_covs_sc,
    verbose=True,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | res_blocks      | ModuleList       | 1.7 K  | train
-------------------------------------------------------------
1.7 K     Trainable params
0         Non-trainable params
1.7 K     Total params
0.007     Total estimated model params size (MB)
52        Modules in train mode
0         Modules in eval mode


/opt/anaconda3/envs/WDN_sim/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 49: 100%|██████████| 24/24 [00:00<00:00, 43.37it/s, train_loss=0.602, val_loss=0.520]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 24/24 [00:00<00:00, 43.33it/s, train_loss=0.602, val_loss=0.520]


TCNModel(output_chunk_shift=0, kernel_size=3, num_filters=8, num_layers=None, dilation_base=2, weight_norm=True, dropout=0.2, input_chunk_length=48, output_chunk_length=1, n_epochs=50, batch_size=32, likelihood=BernoulliLikelihood(prior_p=None, prior_strength=1.0), random_state=0, force_reset=True, pl_trainer_kwargs={'accelerator': 'cpu'})

## Backtest

In [44]:
pred_probs = model.historical_forecasts(
    series=series,                    # Target series (keep as-is, integers 0/1)
    past_covariates=covs_sc, 
    start=0.8,
    forecast_horizon=output_chunk_length,
    stride=1,
    retrain=False,
    last_points_only=True,   # returns a single TimeSeries of 1-step forecasts
    verbose=True,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Predicting DataLoader 0: 100%|██████████| 7/7 [00:00<00:00, 97.29it/s]


In [45]:
actual = series.slice_intersect(pred_probs)
y_true = actual.values().reshape(-1)
y_prob = pred_probs.values().reshape(-1)

# Metrics
brier = brier_score_loss(y_true, y_prob)
roc   = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
ap    = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
print("Brier:", brier, "ROC-AUC:", roc, "Avg Precision:", ap)

Brier: 0.35323383084577115 ROC-AUC: 0.5070652173913043 Avg Precision: 0.20135407866328736
